In [1]:
import pandas as pd 
import torch
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder
from transformers import DistilBertTokenizer,DistilBertForSequenceClassification
from transformers import Trainer,TrainingArguments
from sklearn.metrics import accuracy_score,precision_recall_fscore_support


C:\Users\Vishnu Manikandan\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv('stage1_data.csv')
print(df.head())

                                                text           type
0  I received a traffic challan for speeding how ...  Legal_Request
1  What is the legal process to transfer ownershi...  Legal_Request
2  Can I file a complaint if a car dealer sold me...  Legal_Request
3  What are my rights if I am stopped by traffic ...  Legal_Request
4  How do I apply for a duplicate driving license...  Legal_Request


In [4]:
df = df.dropna()
df = df.drop_duplicates()
df = df[df["text"].str.strip()!=""]

In [5]:
print(df["type"].value_counts())

type
Legal_Request      2593
Illegal_Request    1772
Out_of_Scope       1746
Name: count, dtype: int64


In [6]:
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["type"])

In [7]:
df = df.sample(frac=1).reset_index(drop=True)

In [8]:
df.head()

,text,type,label
0,What are some tips for effective note-taking?,Out_of_Scope,2
1,What is the law on sexual orientation in India?,Legal_Request,1
2,Is there a way to hack food delivery apps for ...,Illegal_Request,0
3,How to bribe a municipal officer for a license?,Illegal_Request,0
4,How can I fake income proof documents?,Illegal_Request,0


In [9]:
train_texts,test_texts,train_labels,test_labels = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42
)

In [10]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

In [11]:
train_encodings = tokenizer(
    train_texts,
    truncation = True,
    padding = True,
    max_length = 128
)

In [12]:
test_encodings = tokenizer(
    test_texts,
    truncation = True,
    padding = True,
    max_length = 128
)

In [13]:
class LawSakthiDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [14]:
train_dataset = LawSakthiDataset(train_encodings,train_labels)
test_dataset = LawSakthiDataset(test_encodings,test_labels)


In [15]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels = 3
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 858.98it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    
    preds = predictions.argmax(axis=1)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted"
    )
    
    acc = accuracy_score(labels, preds)
    
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [29]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    load_best_model_at_end=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [30]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [32]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.077237,0.990188,0.990196,0.990216,0.990188
2,0.012547,0.064910,0.991006,0.990991,0.991010,0.991006
3,0.012547,0.061879,0.991006,0.991000,0.991032,0.991006


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.30it/s]
C:\Users\Vishnu Manikandan\AppData\Roaming\Python\Python310\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.48it/s]
C:\Users\Vishnu Manikandan\AppData\Roaming\Python\Python310\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.40it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=918, training_loss=0.008285811004555563, metrics={'train_runtime': 1234.1381, 'train_samples_per_second': 11.882, 'train_steps_per_second': 0.744, 'total_flos': 334401283144944.0, 'train_loss': 0.008285811004555563, 'epoch': 3.0})

In [33]:
trainer.evaluate()

C:\Users\Vishnu Manikandan\AppData\Roaming\Python\Python310\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.0618789941072464,
 'eval_accuracy': 0.9910057236304171,
 'eval_f1': 0.9909997044208202,
 'eval_precision': 0.9910323242751058,
 'eval_recall': 0.9910057236304171,
 'eval_runtime': 22.808,
 'eval_samples_per_second': 53.621,
 'eval_steps_per_second': 3.376,
 'epoch': 3.0}

In [34]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)

cm = confusion_matrix(test_labels, preds)
print("Confusion Matrix:\n", cm)

print("\nClassification Report:\n")
print(classification_report(test_labels, preds))


C:\Users\Vishnu Manikandan\AppData\Roaming\Python\Python310\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Confusion Matrix:
 [[349   2   3]
 [  0 531   1]
 [  1   4 332]]

Classification Report:

              precision    recall  f1-score   support

           0       1.00      0.99      0.99       354
           1       0.99      1.00      0.99       532
           2       0.99      0.99      0.99       337

    accuracy                           0.99      1223
   macro avg       0.99      0.99      0.99      1223
weighted avg       0.99      0.99      0.99      1223



In [35]:
def predict_query(text, threshold=0.6):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    outputs = model(**inputs)

    probs = torch.nn.functional.softmax(outputs.logits, dim=1)

    predicted_class = torch.argmax(probs).item()
    confidence = torch.max(probs).item()

    label = label_encoder.inverse_transform([predicted_class])[0]

    # Apply threshold
    if confidence >= threshold:
        return label, confidence
    else:
        return "Uncertain", confidence

In [36]:
label, confidence = predict_query("If I delete WhatsApp chats before police check my phone, will that save me from trouble?")
print("Prediction:", label)
print("Confidence:", confidence)

Prediction: Illegal_Request
Confidence: 0.9997525811195374


In [37]:
trainer.save_model("stage1_model")


tokenizer.save_pretrained("stage1_model")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.30it/s]


('stage1_model\\tokenizer_config.json', 'stage1_model\\tokenizer.json')